# 04 - Indexing
## Shakespeare Evidence Desk


## Why This Stage

*In general — what indexing is and why any RAG system needs it.* Once text has been turned into vectors, you still cannot search them efficiently — a pile of vectors is not a search engine. Answering a query means finding the vectors nearest the query's vector, and the naive approach of comparing against every single one does not scale: it is fine for a few thousand items but collapses under millions of comparisons on every request. An **index** is the data structure that makes nearest-neighbor search fast by organizing the vectors so that most of them never have to be examined.

Indexing sits after embedding and before retrieval, because retrieval needs something to search. This is where you decide how the vectors are organized and how much speed you are willing to trade for occasionally missing a near match — the central tension indexing owns, usually resolved with an *approximate nearest-neighbor* (ANN) algorithm and its tuning knobs. Alongside it you settle the supporting choices retrieval depends on: the **distance metric** and the **metadata** each record carries for filtering and citation. Done well, retrieval simply queries a fast, faithful index; done poorly, every query afterward is either too slow to serve or blind to the right results.


- **Goal:** turn the frozen vectors from notebook 03 into a *searchable index* — a structure that answers "which passages are nearest this query?" fast enough to serve, instead of scoring every chunk one by one. This is the last **offline** stage; retrieval (notebook 05) runs on what we build here.
- **Loads from Step 03:** `.artifacts/embeddings.json` (one selected-dimension unit vector per chunk) and `.artifacts/embedding-handoff.json` (the embedding decision). It re-joins each vector to its chunk provenance from `.artifacts/chunks.json` so the index carries the metadata retrieval and citations need. The labeled questions drive the quality proxy.
- **The experiment:** hold the embeddings fixed and vary the **index type** — an *exact* (brute-force) index that scans every vector versus an *approximate nearest-neighbor (ANN)* index with a tunable **speed/recall knob** — measuring build time, index size, search latency, and **indexed recall against exact search** (does the fast index return the same passages the slow-but-correct one would?).
- **Settled alongside:** the **distance metric** (inner product, because notebook 03 stored unit vectors) and the **metadata schema** (which fields each indexed row carries for filtering and provenance).
- **Local stand-in, production target named:** the deployable service indexes into **Azure AI Search (HNSW)**. To keep this notebook self-contained and dependency-free, we demonstrate the *same idea* with a small pure-Python **IVF** index — a different ANN algorithm, but the identical lesson: an index trades exhaustive search for speed via a knob you must tune against measured recall. Section 2's footnote makes the mapping explicit.
- **Not later stages:** this notebook does not retrieve for real, add keyword/hybrid search, rerank, or answer. It exports the index and an indexing decision for notebook 05 (retrieval), where hybrid search and filters are actually run.


## 1. Load the Previous Selection

- **Purpose:** start from notebook 03's exported vectors and decision record, not hidden kernel state. Indexing only arranges vectors for fast lookup — it does not change them — so the embedding selection (dimension, normalization) is fixed input here.
- **What the code does:** locates the repository root, reads `embeddings.json` (the selected-dimension unit vectors) and `chunks.json` (the provenance), **joins them by `chunk_id`** into index records that carry both the vector and the metadata (`work/act/scene/speakers/text`), and builds the **answer key** from the labeled questions (reusing notebook 02/03's citation parser) so Section 4 can grade retrieval.
- **Fails loudly, early:** if `.artifacts/embeddings.json` is absent, run notebook 03 first. If the vectors were built with notebook 03's offline fallback rather than hosted Azure, the notebook **stops here** — an index on fake vectors is meaningless. A vector whose `chunk_id` has no matching chunk is a provenance break and also stops the run.
- **Expected result:** the vector count, the stored dimension and normalization flag, the (hosted) embedding backend, and how many labeled questions resolved to a scene the grader can use. No network call happens here.


In [ ]:
import json
import re
from collections import defaultdict
from pathlib import Path

# Work from the repository root so the notebook and the package read the same .env and artifacts.
ROOT = Path.cwd().resolve()
if not (ROOT / '.env.example').is_file() and (ROOT.parent / '.env.example').is_file():
    ROOT = ROOT.parent
if not (ROOT / '.env.example').is_file():
    raise RuntimeError('Use the repository as the notebook working directory.')

ARTIFACTS = ROOT / '.artifacts'
embeddings_path = ARTIFACTS / 'embeddings.json'
chunks_path = ARTIFACTS / 'chunks.json'
embedding_handoff_path = ARTIFACTS / 'embedding-handoff.json'
questions_path = ARTIFACTS / 'eval-questions.json'
if not embeddings_path.is_file():
    raise RuntimeError('Missing .artifacts/embeddings.json. Run notebook 03 first.')
if not chunks_path.is_file():
    raise RuntimeError('Missing .artifacts/chunks.json. Run notebook 02 first.')

embeddings_payload = json.loads(embeddings_path.read_text(encoding='utf-8'))
chunks_data = json.loads(chunks_path.read_text(encoding='utf-8'))
chunks = chunks_data['chunks'] if isinstance(chunks_data, dict) else chunks_data
embedding_handoff = json.loads(embedding_handoff_path.read_text(encoding='utf-8')) if embedding_handoff_path.is_file() else {}
evaluation_questions = json.loads(questions_path.read_text(encoding='utf-8')) if questions_path.is_file() else {'development': [], 'heldout': []}

stored_dims = embeddings_payload['dims']
normalized = embeddings_payload.get('normalized', True)
embedding_backend = embeddings_payload.get('backend', 'unknown')

# Stop now if the upstream vectors are not real: an index built on offline-fallback vectors is meaningless.
if embedding_backend != 'hosted':
    raise RuntimeError(
        f"embeddings.json was built with the '{embedding_backend}' backend, not hosted Azure. "
        "Re-run notebook 03 against Azure before indexing.")

# Join each vector back to its chunk's provenance by chunk_id: an index row is vector + metadata together.
chunk_by_id = {c['chunk_id']: c for c in chunks}
index_records = []
for vector in embeddings_payload['vectors']:
    chunk = chunk_by_id.get(vector['chunk_id'])
    if chunk is None:
        raise RuntimeError(f'Vector {vector["chunk_id"]} has no matching chunk; re-run notebooks 02 and 03 together.')
    index_records.append({
        'chunk_id': chunk['chunk_id'],
        'work_id': chunk['work_id'],
        'work': chunk['work'],
        'scenes': chunk['scenes'],
        'speakers': chunk.get('speakers', []),
        'text': chunk['text'],
        'embedding': vector['embedding'],
    })

wrong_dims = [r['chunk_id'] for r in index_records if len(r['embedding']) != stored_dims]
if wrong_dims:
    raise RuntimeError(f'{len(wrong_dims)} vector(s) are not {stored_dims}-d; re-export from notebook 03.')

work_titles = {r['work_id']: r['work'] for r in index_records}
print(f'Loaded {len(index_records):,} vectors at {stored_dims} dims (normalized={normalized}) from the {embedding_backend} backend.')
print(f'Joined every vector to chunk provenance across {len(work_titles)} works.')

# ---- answer key: turn each labeled question into the set of index rows holding its cited scene ----
ROMAN_VALUES = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}


def roman_to_int(value):
    # Normalize a scene number written as a Roman numeral or digits ('V' or '5' -> 5).
    if value is None:
        return None
    text = str(value).strip().upper()
    if text.isdigit():
        return int(text)
    total, previous = 0, 0
    for char in reversed(text):
        if char not in ROMAN_VALUES:
            return None
        current = ROMAN_VALUES[char]
        total += -current if current < previous else current
        previous = max(previous, current)
    return total or None


lowercase_title = {work_id: title.lower() for work_id, title in work_titles.items()}


def cited_locations(question):
    # Read the label's citation -> list of (work, act, scene); uses the label only, never the question's words.
    label_text = (question.get('question', '') + ' ' + question.get('expected_evidence', '')).lower()
    works = [work_id for work_id, title in lowercase_title.items() if title in label_text]
    act_scene_pairs = re.findall(r'act\s+(\d+|[ivxlc]+)\s*,?\s*scene\s+(\d+|[ivxlc]+)',
                                 question.get('expected_evidence', '').lower())
    acts_scenes = [(roman_to_int(act), roman_to_int(scene)) for act, scene in act_scene_pairs]
    return [(work_id, act, scene) for work_id in works for act, scene in acts_scenes]


def row_in_scene(record, work_id, act, scene):
    return record['work_id'] == work_id and any(a == act and s == scene for a, s in record['scenes'])


# For each answerable question, the set of row indices whose scene the label points to.
answer_key = []
for question in evaluation_questions.get('development', []):
    if not question.get('answerable'):
        continue
    locations = [(w, a, s) for w, a, s in cited_locations(question) if a is not None and s is not None]
    relevant_rows = {i for i, record in enumerate(index_records)
                     for (w, a, s) in locations if row_in_scene(record, w, a, s)}
    if relevant_rows:
        answer_key.append((question, relevant_rows))

print(f'Answer key: {len(answer_key)} labeled dev question(s) resolved to an indexed scene.')


## 2. Explain the Alternatives

An index is just a way of organizing the vectors so that, given a query, we don't have to compare it to *every* passage. A choice is **good** when it returns the same passages an exhaustive search would, but much faster and without ballooning storage. We **fix the embeddings** (dimension, normalization) and tune how the vectors are arranged and searched.

**The choices, in easy terms:**

| Choice | What it means (plain) | Why it matters |
|---|---|---|
| **Index type** *(our main experiment)* | *Exact* checks the query against all N passages (always correct, cost grows with N). *ANN* groups passages so the query only checks a promising fraction. | Exact is perfect but slow at scale; ANN is the only thing that stays fast as the corpus grows — but it can **miss** a close passage. We measure how much it misses. |
| **ANN knob** *(`nprobe`)* | How many passage groups the query is allowed to open before giving up. Low = very fast, risk missing the right passage; high = slower, closer to exact. | This is the dial that trades **speed for recall**. The whole experiment is finding the lowest setting that still finds what exact search finds. |
| **Build knob** *(`nlist`)* | How many groups the passages are split into when the index is built. | More groups = each group is smaller so a query scans fewer passages, but the grouping step (build) costs more. A one-time cost paid to make every later query cheap. |
| **Distance metric** | How "nearness" is scored — here the **inner product** (dot). | Notebook 03 stored **unit** vectors, so inner product *equals* cosine. Settled: use inner product, get cosine ranking for free. |
| **Metadata schema** | Which fields each indexed row carries besides the vector: key (`chunk_id`), filterable (`work_id`), and retrievable provenance (`work/act/scene/speakers/text`). | Retrieval filters ("only *Hamlet*") and citations need these fields *in the index*; forgetting one here breaks notebook 05 and the final answer's sourcing. |
| **Hybrid (keyword + vector)** | Also match on literal words (BM25), not only meaning, then fuse the two rankings. | A real quality lever for exact quotes and rare names — but it belongs to **retrieval** (notebook 05), where the query exists. Named here, measured there. |

- **Why index type is the experiment we actually run:** it is the choice with a genuine, measurable tradeoff at this stage — correctness (recall vs exact) against speed and build cost. Metric and schema are settled decisions, not experiments.
- **Honest expectation:** at ten plays the corpus is small enough that *exact search is already fast*, so ANN's speed win is modest here. We still run the comparison to (a) prove the ANN index returns the same passages within tolerance, and (b) establish the config that will matter when the corpus grows — because the production index (Azure AI Search) is ANN by necessity.

> **Footnote — our local IVF vs. production Azure AI Search (HNSW).** Both are **approximate nearest-neighbor** indexes solving the same problem: *don't compare the query to every vector.* They differ in how they prune. **IVF** (what we run here, in pure Python) clusters the vectors into `nlist` groups and, at query time, only opens the `nprobe` groups whose centers are closest — so it scans roughly `nprobe/nlist` of the corpus. **HNSW** (what Azure AI Search uses) instead builds a layered graph and walks greedily toward the query, with `M` (graph connectivity) and `efConstruction`/`efSearch` as its build/search knobs. The *algorithm* differs, but the *lesson is identical*: an ANN index has a **speed-for-recall dial** you must tune against measured recall, and a **build cost** you pay once. We use IVF only because it is faithful, dependency-free, and easy to read; when the service indexes for real, the same numbers (recall vs exact, latency, build time) are re-measured on HNSW.

### Where each choice lives in the code

| Choice | Implemented by | Measured / checked by | Status |
|---|---|---|---|
| **Index type** | `build_ivf` + `ivf_search` vs `exact_search` — Section 3 | `recall_vs_exact` + latency timing (`index_metrics`) — Section 4 | implemented **+ measured + decided** |
| **ANN knob (`nprobe`)** | the `nprobe` argument to `ivf_search` — Section 3 | the `nprobe` sweep in `index_metrics` — Section 4 | implemented **+ measured** |
| **Build knob (`nlist`)** | `NLIST` + `build_ivf` — Section 3 | build-duration printout — Section 3 | implemented, cost observed |
| **Distance metric** | `dot` on unit vectors — Section 3 | the retrieval proxy agreeing with notebook 03 — Section 4 | settled (inner product == cosine) |
| **Metadata schema** | `METADATA_SCHEMA` + the index records — Sections 1, 3 | exported and consumed by notebook 05 | settled |
| **Hybrid (keyword + vector)** | — none (vector-only here) | — none | **gap: described, run in notebook 05** |


> ### Footnote — ANN index families, a FAQ for reread
>
> Background built up while working through this stage. It is **conceptual grounding**, not part of the experiment — skim it the first time, reread it when the terms blur.
>
> **Q: Are IVF and HNSW "indexing" or "search" techniques?** Both. An ANN method is inseparably a *structure built offline* (the index) and a *procedure run at query time* (the search) — HNSW's graph **is** the thing you traverse. The notebook files them under **Indexing** because the decisions (algorithm, build/search knobs, metric, schema) are made when building. That is a different layer from **retrieval** (notebook 05), which is the query *strategy* — keyword/BM25, dense, hybrid + RRF, filters, top-k. Dense retrieval *uses* an ANN index as its engine; BM25 uses a separate text index; hybrid runs both and fuses.
>
> **Q: Which of these use graphs?** Only HNSW.
>
> | Technique | Structure | Graph? |
> |---|---|---|
> | **HNSW** | layered "small-world" **graph** walked toward the query | ✅ yes |
> | **IVF** | **clusters** (centroids + inverted lists); open the nearest clusters | ❌ partitioning |
> | **IVF-PQ** | IVF clustering **+ PQ compression** of the vectors | ❌ clustering + quantization |
>
> Families: **graph-based** → HNSW, DiskANN/Vamana · **partition/clustering** → IVF, IVF-PQ · **quantization/compression** → PQ (an add-on, usually paired with IVF).
>
> **Q: Is IVF / IVF-PQ "just KNN"?** No — they *approximate* KNN. KNN is the *task* ("return the k nearest"); **exact/flat** KNN is the exhaustive solution (check everything, always correct). **IVF** approximates *which* vectors to check; **IVF-PQ** approximates *which* vectors **and** *the distances* (compressed codes). That is exactly what `recall_vs_exact` measures — if IVF were "just KNN" that number would always be 100%; the point is that it usually isn't.
>
> **Q: How does IVF work?** *Build once:* k-means splits the vectors into `nlist` clusters → centroids + an inverted list per cluster. *Per query:* score the query against the centroids, open the `nprobe` nearest clusters, exact-score only their members, return top-k.
> *Mini example* — 8 two-D vectors split into `nlist=2`: an "x-heavy" cluster {A,B,E,G} and a "y-heavy" cluster {C,D,F,H}. Query `[0.85,0.15]` with `nprobe=1` scores the two centroids (`0.74` vs `0.26`), opens **only the x-heavy cluster**, scans 4 vectors instead of 8, returns **E**. It never looked at the other cluster — the speedup, and the place a miss can happen. `nprobe=2` would scan all 8 = exact.
>
> **Q: How does PQ (and IVF-PQ) work?** PQ *compresses*: chop each vector into `m` sub-vectors; for each slot learn a codebook of representative sub-vectors; replace each sub-vector with the **id** of its nearest codebook entry. A `dim=8`, `m=2` vector becomes a 2-id code — ~2 bytes instead of 32. Distances use a precomputed table (query-half vs each codebook entry), so a vector's approximate score is a couple of table lookups and an add — vectors are never decompressed. **IVF-PQ** = IVF narrows *which* clusters to scan, PQ makes scoring/storage *cheap* inside them. Standard recipe for billion-scale, memory-capped indexes.
>
> **Q: Isn't HNSW just always superior?** No — it wins on **recall-vs-latency for in-memory, mid-to-large** datasets (why Azure AI Search defaults to it), but pays in **memory** (full vectors + a big edge graph) and **build/update cost** (slow build; deletes degrade the graph). At billion-scale under a memory cap, **IVF-PQ** wins; at a few thousand vectors, **exact/flat** wins. "Best" is always relative to the constraint (memory, size, update rate, disk vs RAM).
>
> **Q: So what is *this notebook* comparing?** Not IVF vs HNSW (it has no HNSW engine). It compares **exact (brute-force)** vs **IVF at `nprobe` = 1/2/4/8** on recall-vs-exact, latency, and size. The goal is a **transferable procedure** — *"exact is the correctness ceiling; find the cheapest approximate setting whose recall-vs-exact clears tolerance"* — which is identical whether the dial is IVF's `nprobe` or HNSW's `efSearch`. IVF is used here only because it is pure-Python, dependency-free, and easy to read; production applies the same method to HNSW.


## 3. Run Controlled Experiments

- **Fixed upstream:** the vectors, their dimension, and the query set are held constant; only the index type and its knobs change between candidates.
- **What the code does:** declares the **metadata schema** (the fields every indexed row carries), builds the **exact** index (just the vectors — search scans all of them), then builds **one IVF** ANN index by clustering the vectors into `nlist` groups (a short, deterministic k-means), timing the build. The `nprobe` dial is applied at *search* time, so the ANN variants cost nothing extra to build.
- **Query embedding:** to grade retrieval we embed the labeled *questions* with the same keyless Azure deployment and on-disk cache as notebook 03, then truncate + L2-normalize to the stored dimension so a query vector lives in the same space as the indexed vectors. Because notebook 03 already embedded these questions, the cache usually serves them with **0 API calls**.
- **No offline fallback — fail fast:** building and searching the index is pure local compute. Question embedding reads the on-disk cache first; if a question is *not* cached and Azure is unreachable, `embed_query` raises and the notebook stops with the Azure error instead of fabricating vectors. Fix credentials / `.env` and re-run.
- **Expected result:** the schema fields, the exact index size, the IVF build time and cluster count (`nlist`), and the `nprobe` values to sweep.

> **Note — why the IVF build is kept small.** The pure-Python k-means here runs only a few iterations over `nlist ≈ √N` clusters: enough to form sensible groups for a ten-play corpus, cheap enough to run in a notebook. It is **illustrative**, not production-tuned — Azure AI Search builds its HNSW graph far more efficiently. The point is to expose the *shape* of the tradeoff (build cost up front, faster queries after), not to benchmark a pure-Python ANN against a real engine.


In [ ]:
import hashlib
import math
import time

from dotenv import dotenv_values
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

# The metadata schema: what each indexed row carries beyond the vector, and the role each field plays.
# Notebook 05 (retrieval) and the final citations depend on these being present in the index.
METADATA_SCHEMA = {
    'chunk_id': 'key',                 # unique row id, joins back to the source chunk
    'work_id': 'filterable',           # restrict search to a play, e.g. only Hamlet
    'work': 'retrievable',             # human title for citations
    'scenes': 'retrievable',           # (act, scene) pairs for provenance
    'speakers': 'retrievable',         # who speaks, for citations
    'text': 'retrievable',             # the passage itself, shown as evidence
    'embedding': 'vector',             # the searchable vector field
}

vectors = [r['embedding'] for r in index_records]
N = len(vectors)


def dot(a, b):
    # Score between two unit vectors; equals cosine similarity. Higher = more relevant.
    return sum(x * y for x, y in zip(a, b))


def l2_normalize(vector):
    length = math.sqrt(sum(value * value for value in vector))
    return [value / length for value in vector] if length else vector


# ---- exact (flat) index: no build step; a search scans every vector ----
def exact_search(query_vector, k):
    return sorted(range(N), key=lambda i: dot(query_vector, vectors[i]), reverse=True)[:k]


# ---- IVF (ANN) index: cluster the vectors into nlist groups; a query opens only the nprobe nearest groups ----
NLIST = min(round(math.sqrt(N)) or 1, 48)   # number of clusters (~sqrt N, capped); build knob (HNSW's efConstruction analog)
KMEANS_ITERS = 3                            # k-means passes: a few are enough in pure Python
NPROBE_SWEEP = [n for n in (1, 2, 4, 8) if n <= NLIST]  # clusters a query may open; the speed/recall dial (efSearch analog)


def build_ivf(data, nlist, iterations):
    # Short deterministic k-means: evenly spaced seeds, assign to nearest centroid, recompute, repeat.
    step = max(1, len(data) // nlist)
    centroids = [l2_normalize(data[(c * step) % len(data)][:]) for c in range(nlist)]
    cluster_of = [0] * len(data)
    for _ in range(iterations):
        for i, vector in enumerate(data):
            cluster_of[i] = max(range(nlist), key=lambda c: dot(vector, centroids[c]))
        totals = [[0.0] * len(data[0]) for _ in range(nlist)]
        member_count = [0] * nlist
        for i, vector in enumerate(data):
            c = cluster_of[i]
            member_count[c] += 1
            for d, value in enumerate(vector):
                totals[c][d] += value
        for c in range(nlist):
            if member_count[c]:
                centroids[c] = l2_normalize([value / member_count[c] for value in totals[c]])
    rows_by_cluster = defaultdict(list)
    for i, c in enumerate(cluster_of):
        rows_by_cluster[c].append(i)
    return centroids, dict(rows_by_cluster)


build_started = time.perf_counter()
ivf_centroids, ivf_lists = build_ivf(vectors, NLIST, KMEANS_ITERS)
ivf_build_ms = (time.perf_counter() - build_started) * 1000


def ivf_search(query_vector, k, nprobe):
    # Open only the nprobe nearest clusters, then score exactly within that smaller candidate pool.
    nearest_clusters = sorted(range(len(ivf_centroids)),
                              key=lambda c: dot(query_vector, ivf_centroids[c]), reverse=True)[:nprobe]
    candidate_rows = [i for c in nearest_clusters for i in ivf_lists.get(c, ())]
    return sorted(candidate_rows, key=lambda i: dot(query_vector, vectors[i]), reverse=True)[:k]


# ---- embed the labeled questions with the hosted Azure deployment (keyless), cached on disk ----
# No offline fallback: if a question is not cached and Azure is unreachable, embed_query raises and the
# notebook stops. Grading an index with fabricated query vectors would be worthless.
config = {key: value for key, value in dotenv_values(ROOT / '.env').items() if value}
token_provider = get_bearer_token_provider(DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default')
client = AzureOpenAI(
    azure_endpoint=config['AZURE_OPENAI_ENDPOINT'],
    api_version=config['AZURE_OPENAI_API_VERSION'],
    azure_ad_token_provider=token_provider,
)
deployment = config['AZURE_OPENAI_EMBEDDING_DEPLOYMENT']

CACHE_DIR = ARTIFACTS / 'embedding-cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
query_api_calls = 0


def embed_query(text):
    # Reuse notebook 03's on-disk cache when possible; otherwise call Azure once and cache the vector.
    global query_api_calls
    cache_file = CACHE_DIR / (hashlib.sha256(f'{deployment}|{text}'.encode('utf-8')).hexdigest() + '.json')
    if cache_file.is_file():
        full_vector = json.loads(cache_file.read_text())
    else:
        full_vector = client.embeddings.create(model=deployment, input=[text]).data[0].embedding
        cache_file.write_text(json.dumps(full_vector))
        query_api_calls += 1
    return l2_normalize(full_vector[:stored_dims])   # match the indexed vectors' space (truncate + renormalize)


# Embed every answer-key question once; the measurement loop reuses these vectors.
query_vectors = [embed_query(question['question']) for question, _ in answer_key]

exact_index_mb = N * stored_dims * 4 / (1024 * 1024)
print(f'Metadata schema fields: {", ".join(METADATA_SCHEMA)}')
print(f'Exact index: {N:,} vectors × {stored_dims} dims = {exact_index_mb:.2f} MB (float32), no build step.')
print(f'IVF index: built {NLIST} clusters in {ivf_build_ms:.0f} ms ({KMEANS_ITERS} k-means iters).')
print(f'nprobe sweep: {NPROBE_SWEEP} (each opens that many of {NLIST} clusters).')
print(f'Embedded {len(query_vectors)} question(s) for grading; {query_api_calls} Azure call(s) (rest from cache).')


## 4. Measure and Inspect

- **Four lenses:** indexed recall against exact search (the one that matters for an index), search latency, index size, and the end-to-end retrieval proxy.
- **Indexed recall vs exact — the index's own correctness test.** Exact search is the ground truth here: for each query it returns the *true* top-k nearest passages. We then ask the ANN index the same query and measure **what fraction of exact's top-k it also returned** (`recall_vs_exact`). An exact index scores 1.00 by definition; an ANN index that opens too few clusters scores lower because it never looked at the cluster holding a near passage. This is distinct from answer quality — it measures *fidelity to exhaustive search*.
- **Search latency:** average milliseconds per query, measured over the answer-key questions. Exact scans all `N` vectors every time; IVF scans only the vectors inside the opened clusters, so lower `nprobe` is faster. At ten plays both are quick — the gap widens with corpus size.
- **Index size:** bytes to hold the vectors (`N × dims × 4`, float32) plus, for IVF, the cluster centroids. IVF is *slightly larger* than exact (it adds centroids); its payoff is **speed**, not storage. We report size honestly so nobody expects ANN to shrink the index.
- **Retrieval-quality proxy (recall@1 / recall@5 / MRR):** the same label-graded nearest-neighbor check as notebook 03, but run *through each index*. It confirms that choosing an ANN config does not quietly drop the right passage below where generation can use it. Exact is the quality ceiling; a good ANN config matches it.
- **Honest limit:** the proxy and the recall-vs-exact average use only the labeled dev questions that resolve to a parsed scene, so the sample is small. This pre-screens index configs; the end-to-end judges are retrieval (notebook 05) and continuous evaluation (notebook 10).

> **Footnote — "recall vs exact" in plain terms.** Suppose a query's *true* five nearest passages (what exact search returns) are **{A, B, C, D, E}**. A fast ANN index that only opened a couple of clusters might return **{A, B, C, D, Z}** — it found four of the five but missed **E** (E lived in a cluster it never opened) and padded with **Z**. That is **recall_vs_exact = 4/5 = 0.80**: the index agreed with exhaustive search 80% of the time. Turn the `nprobe` dial up and it opens more clusters, eventually recovering **E** and reaching 1.00 — but each extra cluster opened costs latency. The experiment is simply: **what is the lowest `nprobe` that still scores ~1.00?**


In [ ]:
K = 5  # top-k cutoff for recall-vs-exact and the retrieval proxy (matches notebook 03)


def grade_ranking(ranked_rows, relevant_rows):
    # Find where the first relevant row landed, and return (hit@1, hit@K, reciprocal_rank).
    position = next((p for p, row in enumerate(ranked_rows) if row in relevant_rows), None)
    if position is None:
        return 0, 0, 0.0
    return int(position == 0), int(position < K), 1.0 / (position + 1)


def evaluate_index(search):
    # Run one index's `search` over every answer-key question: time it, grade retrieval, and
    # compare its top-K to exact search's top-K. `search` is exact_search or an ivf_search variant.
    top1_hits = topk_hits = 0
    reciprocal_total = recall_total = 0.0
    started = time.perf_counter()
    for query_vector, (_, relevant_rows) in zip(query_vectors, answer_key):
        ranked_rows = search(query_vector)
        exact_top = set(exact_search(query_vector, K))
        recall_total += len(set(ranked_rows[:K]) & exact_top) / K
        hit1, hitk, reciprocal = grade_ranking(ranked_rows, relevant_rows)
        top1_hits += hit1
        topk_hits += hitk
        reciprocal_total += reciprocal
    questions = max(len(answer_key), 1)
    return {
        'recall@1': top1_hits / questions,
        f'recall@{K}': topk_hits / questions,
        'mrr': reciprocal_total / questions,
        'recall_vs_exact': recall_total / questions,
        'latency_ms': (time.perf_counter() - started) * 1000 / questions,
    }


# The indexes to compare: exact baseline, then IVF at each probe count. Return >= K rows to score MRR fairly.
TOP_N = max(K, 10)
candidates = {'exact': lambda q: exact_search(q, TOP_N)}
for nprobe in NPROBE_SWEEP:
    candidates[f'ivf(nprobe={nprobe})'] = (lambda q, p=nprobe: ivf_search(q, TOP_N, p))

index_metrics = {name: evaluate_index(search) for name, search in candidates.items()}

# Index size in bytes: every index stores the vectors; IVF adds the cluster centroids.
vector_bytes = N * stored_dims * 4
centroid_bytes = NLIST * stored_dims * 4
sizes = {name: vector_bytes + (centroid_bytes if name.startswith('ivf') else 0) for name in candidates}

print(f'Measured over {len(answer_key)} resolved dev question(s); exact search is the recall-vs-exact ground truth.\n')
header = ('index', 'size MB', 'latency ms', 'rec vs exact', 'recall@1', f'recall@{K}', 'mrr')
print('{:<18} {:>8} {:>11} {:>13} {:>9} {:>9} {:>7}'.format(*header))
for name, m in index_metrics.items():
    print('{:<18} {:>8.2f} {:>11.2f} {:>12.0%} {:>8.0%} {:>8.0%} {:>7.3f}'.format(
        name, sizes[name] / (1024 * 1024), m['latency_ms'], m['recall_vs_exact'],
        m['recall@1'], m[f'recall@{K}'], m['mrr']))

print(f'\nIVF build cost: {ivf_build_ms:.0f} ms (one-time, {NLIST} clusters).')


## 5. Justify the Selection

- **Decision rule (data-driven, deterministic):** among the candidate indexes, keep only those whose **recall vs exact** and **retrieval proxy** (MRR, Recall@5) stay within tolerance (`0.02`) of the exact baseline — i.e. indexes that behave like exhaustive search — then pick the one with the **lowest search latency**. We buy speed only where it costs no measurable recall.
- **Why lowest-latency-within-tolerance:** exact search is correct but its cost grows with the corpus; an ANN config that matches exact's answers while scanning a fraction of the vectors is strictly better *at scale*. The rule prefers the fastest such config, and transparently falls back to **exact** when (as can happen on a small corpus) no ANN setting is both faithful *and* faster.
- **Metric and schema are settled separately:** we score with the **inner product** because notebook 03 stored unit vectors (so it equals cosine), and we carry the **metadata schema** (key / filterable / retrievable / vector) into the index so notebook 05 can filter and cite. Neither is an experiment.
- **Production mapping stated, not hidden:** the selected IVF config is the *local* answer. The deployable service indexes into **Azure AI Search (HNSW)**; the handoff records the chosen behavior (near-exact recall at reduced scan) and the schema so the production index is configured to the same target, then re-measured with HNSW's own `M` / `efSearch`.
- **Inputs are guaranteed real:** Section 1 already stopped if the embeddings were built offline, so this selection always reflects hosted-Azure vectors — there is no fallback caveat to track.
- **Expected result:** the selected index with the numbers that justify it (recall vs exact, latency, size), the metric and schema decisions, and the rejected candidates noted with their measured gap.


In [ ]:
TOLERANCE = 0.02   # max acceptable drop vs exact on recall-vs-exact, MRR, and Recall@K
metric = 'inner_product'  # unit vectors from notebook 03, so dot == cosine

baseline = index_metrics['exact']


def matches_exact(metrics):
    # True if this index behaves like exhaustive search: same passages, same grades, within tolerance.
    return ((1.0 - metrics['recall_vs_exact']) <= TOLERANCE
            and (baseline['mrr'] - metrics['mrr']) <= TOLERANCE
            and (baseline[f'recall@{K}'] - metrics[f'recall@{K}']) <= TOLERANCE)


# Keep the indexes that match exact within tolerance, then pick the fastest. Exact always qualifies.
eligible = [name for name, metrics in index_metrics.items() if matches_exact(metrics)]
selected_index = min(eligible, key=lambda name: index_metrics[name]['latency_ms'])
selected = index_metrics[selected_index]
rejected = [name for name in index_metrics if name != selected_index]

speedup = baseline['latency_ms'] / selected['latency_ms'] if selected['latency_ms'] else 1.0

print(f'Selected index: {selected_index} (metric={metric}, nlist={NLIST})')
print(f"  recall vs exact: {selected['recall_vs_exact']:.0%} | latency {selected['latency_ms']:.2f} ms/query "
      f"({speedup:.2f}× vs exact)")
print(f"  retrieval proxy recall@1 / recall@{K} / mrr: "
      f"{selected['recall@1']:.0%} / {selected[f'recall@{K}']:.0%} / {selected['mrr']:.3f}")
print(f"  vs exact baseline: {baseline['recall@1']:.0%} / {baseline[f'recall@{K}']:.0%} / {baseline['mrr']:.3f}")
print('\nRejected:')
for name in rejected:
    metrics = index_metrics[name]
    gap = max(1.0 - metrics['recall_vs_exact'],
              baseline['mrr'] - metrics['mrr'],
              baseline[f'recall@{K}'] - metrics[f'recall@{K}'])
    reason = 'recall below tolerance' if gap > TOLERANCE else 'slower than the selection'
    print(f"  {name}: rec-vs-exact={metrics['recall_vs_exact']:.0%} latency={metrics['latency_ms']:.2f} ms "
          f"(Δ={gap:+.3f}) — {reason}")


## 6. Export the Handoff

- **What gets exported:**
  - `.artifacts/index.json` — the searchable index notebook 05 loads: the selected index type and metric, the ANN parameters (`nlist`, chosen `nprobe`), the **metadata schema**, every indexed row (`chunk_id`, `work_id`, provenance, and the vector), and the IVF structure (centroids + inverted lists) so retrieval can reproduce the same search without rebuilding.
  - `.artifacts/indexing-handoff.json` — the decision record: index type, metric, parameters, schema, build time, index size, and the full candidate comparison (recall vs exact, latency, proxy) that justified the choice, plus a back-reference to the embedding backend and dimension.
- **Provenance preserved:** each row keeps its `chunk_id` and scene metadata, so retrieval, reranking, generation, and citations downstream still resolve back through the chunk to the original source lines.
- **Production mapping recorded:** the handoff notes the production target (Azure AI Search / HNSW) and the behavior to reproduce there (near-exact recall, the metadata schema), so the deployable index is configured to the same contract and re-measured with its own knobs.
- **Hygiene:** `.artifacts/` is gitignored and notebook outputs are stripped on commit — do not commit the index (it contains the vectors) or the decision record.
- **Expected result:** both artifacts written, with the selected index, metric, parameters, row count, and size printed.


In [ ]:
index_path = ARTIFACTS / 'index.json'
indexing_handoff_path = ARTIFACTS / 'indexing-handoff.json'

selected_is_ivf = selected_index.startswith('ivf')
selected_nprobe = int(re.search(r'nprobe=(\d+)', selected_index).group(1)) if selected_is_ivf else None

# index.json: everything notebook 05 needs to search without rebuilding — config, schema, rows, and the IVF structure.
index_payload = {
    'index_type': 'ivf' if selected_is_ivf else 'exact',
    'metric': metric,
    'dims': stored_dims,
    'normalized': normalized,
    'embedding_backend': embedding_backend,
    'params': {'nlist': NLIST, 'nprobe': selected_nprobe} if selected_is_ivf else {},
    'schema': METADATA_SCHEMA,
    'records': index_records,  # chunk_id, work_id, work, scenes, speakers, text, embedding
    'ivf': ({'centroids': ivf_centroids,
             'lists': {str(cluster): rows for cluster, rows in ivf_lists.items()}} if selected_is_ivf else None),
    'production_target': {'service': 'azure-ai-search', 'algorithm': 'hnsw'},
}
index_path.write_text(json.dumps(index_payload), encoding='utf-8')  # compact: records carry the vectors

# indexing-handoff.json: the decision record and the measured comparison that justified it.
indexing_handoff = {
    'stage': '04-indexing',
    'index_type': index_payload['index_type'],
    'metric': metric,
    'params': index_payload['params'],
    'schema': METADATA_SCHEMA,
    'record_count': len(index_records),
    'dims': stored_dims,
    'index_bytes': sizes[selected_index],
    'build_ms': ivf_build_ms if selected_is_ivf else 0.0,
    'tolerance': TOLERANCE,
    'candidates': index_metrics,            # recall-vs-exact, latency, and proxy for every candidate
    'selected': selected_index,
    'embedding_backend': embedding_backend,
    'resolved_questions': len(answer_key),
    'production_target': index_payload['production_target'],
}
indexing_handoff_path.write_text(json.dumps(indexing_handoff, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Wrote {len(index_records):,}-row {index_payload["index_type"]} index to {index_path.relative_to(ROOT)} '
      f'({sizes[selected_index] / (1024 * 1024):.2f} MB).')
print(f'Wrote decision record to {indexing_handoff_path.relative_to(ROOT)}')
print(f'Selected: {selected_index} | metric={metric} | params={index_payload["params"]}')


## Observations and Open Questions

Honest notes from working through this stage — what the numbers do and do not prove, and what to revisit later.

- **"Recall vs exact" is the metric that actually belongs to indexing.** Notebook 03 asked *is this vector good?*; this notebook asks *does the fast index return the same passages the slow, correct one would?* Those are different questions. An ANN index can have perfect embeddings and still miss a near passage simply because it never opened the right group — that is what `recall_vs_exact` catches, and it is why we grade against exact search rather than against the labels alone.

- **At ten plays, ANN's speed win is small — and we say so.** The corpus is small enough that exact search is already fast, so the honest outcome may be that exact (or a high `nprobe`) is selected. That is fine: the experiment's job here is to *prove the ANN path is faithful* and establish the config, so that when the corpus grows (or moves to Azure AI Search), the speed advantage is real and already validated. Don't read the current latency gap as the whole story.

- **IVF here is a stand-in for HNSW in production.** The algorithms differ (clustering vs a navigable graph), so the exact `nlist`/`nprobe` values do not transfer to Azure AI Search's `M`/`efSearch`. What transfers is the *procedure*: pick the cheapest search setting whose recall-vs-exact clears tolerance, then re-measure it on the real engine. Treat the selected IVF config as a design decision, not a tuned production parameter.

- **The index is bigger than the raw vectors, not smaller.** IVF adds centroids; HNSW adds graph edges. Indexing buys **speed**, never storage — if anything the index costs a little more memory than the vectors alone. We report size so this tradeoff is explicit and nobody expects an index to compress the corpus.

- **The metadata schema is a contract, not an afterthought.** Every field carried here (`work_id` for filters, `scenes`/`speakers`/`text` for citations) is something a later stage depends on. Dropping one to save space would silently break notebook 05's filters or the final answer's sourcing, so the schema is exported explicitly and consumed verbatim.

- **Hybrid search is deliberately absent.** Keyword/BM25 fusion is a real lever for exact quotes and rare names, but it needs the *query*, which lives in notebook 05. We named it here and will measure it there rather than pretending a vector-only index is the final word on retrieval.

- **How we will revisit.** The index is a swappable layer: change the index type or knobs (or move to Azure AI Search), re-export `index.json`, and re-run 05→08 on the **same** questions. Iterate on the **development** set; check the **held-out** set only periodically so the evaluation stays honest.


## Completion Check and Next Notebook

- **Restart and run in order:** restart the kernel and run top to bottom to confirm this notebook depends only on the notebook 02/03 artifacts, not hidden state. Building and searching the index is pure local compute; only question embedding can touch Azure, and it is served from notebook 03's cache when available.
- **What we achieved:** every vector joined back to its chunk provenance into a schema'd index row; an exact index and an IVF ANN index built and timed; index types compared on **recall vs exact**, latency, size, and the label-graded retrieval proxy; the fastest faithful index selected by a deterministic rule; the distance metric (inner product) and metadata schema settled; and two exported artifacts (`index.json`, `indexing-handoff.json`).
- **Honest limits:** the corpus is small, so ANN's speed advantage is modest and the proxy sample is tiny; the local IVF is a stand-in for the production Azure AI Search (HNSW) index, and hybrid (keyword + vector) search is deferred to notebook 05. The real judges are retrieval (notebook 05) and continuous evaluation (notebook 10).
- **Privacy and hygiene:** outputs are stripped on commit by `nbstripout`; `.artifacts/` (including `index.json`, which contains the vectors) is ignored. Do not commit the index or the decision record.
- **Next stage — 05 Retrieval:** load `.artifacts/index.json`, run keyword/BM25, dense, and hybrid (reciprocal rank fusion) retrieval over the index, apply top-k and metadata filters, and measure Recall@k, MRR, empty-result rate, filter correctness, and p95 latency while holding this index fixed.
